# Qwen3.8-27B resolution sweep: raw versus gray220

Full 21-page OCR output is retained in this notebook for every run. The matrix is six longest-edge settings (`1024`, `1400`, `1600`, `1800`, `2000`, `2200`) crossed with original raw images and `gray220` images. Per-page preprocessing audit, GPU snapshots, benchmark configuration, timing, and Golden scores are captured as cell output.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time
import urllib.request

ROOT = Path(".")
AUDIT_PYTHON = Path("python")
GOLDEN = ROOT / "results/bot_credit_bureau_21p_golden_transcript_20260829.json"
SWEEP_ROOT = ROOT / "inputs/bot_credit_bureau_2559/resolution_sweep"
MANIFEST = SWEEP_ROOT / "gray220_resolution_sweep_audit.json"
MODEL = "qwen3.8-27b"
ENDPOINT = "http://127.0.0.1:8000"
PROMPT_PROFILE = "document"
CONCURRENCY = 7
SERVER_MAX_NUM_SEQS = 7
MTP_TOKENS = 0
DISABLE_THINKING = True
SIZES = (1024, 1400, 1600, 1800, 2000, 2200)
TEMPERATURE = 0.0
TOP_P = 0.8
TOP_K = 20
REPETITION_PENALTY = 1.05
PRESENCE_PENALTY = 0.0
MAX_TOKENS = 8192

def gpu_snapshot():
    return subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()

def wait_for_model(timeout_seconds=900):
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(ENDPOINT + "/v1/models", timeout=5) as response:
                payload = json.load(response)
            print("Endpoint ready:", ", ".join(item["id"] for item in payload.get("data", [])))
            return
        except Exception as error:
            last_error = error
            time.sleep(5)
    raise TimeoutError(f"Endpoint did not become ready: {last_error}")

def run_ocr(label, image_directory, result_path):
    output = ROOT / result_path
    if output.exists():
        raise FileExistsError(f"Refusing to overwrite {output}")
    command = [
        str(AUDIT_PYTHON), "benchmark/ocr_benchmark.py", str(ROOT / image_directory), str(output),
        "--endpoint", ENDPOINT,
        "--model", MODEL,
        "--prompt-profile", PROMPT_PROFILE,
        "--temperature", str(TEMPERATURE),
        "--top-p", str(TOP_P),
        "--top-k", str(TOP_K),
        "--repetition-penalty", str(REPETITION_PENALTY),
        "--presence-penalty", str(PRESENCE_PENALTY),
        "--max-tokens", str(MAX_TOKENS),
        "--concurrency", str(CONCURRENCY),
        "--images-per-request", "1",
        "--server-max-num-seqs", str(SERVER_MAX_NUM_SEQS),
    ]
    if DISABLE_THINKING:
        command.append("--disable-thinking")
    print(f"\n=== {label} ===")
    print("GPU before:\n" + gpu_snapshot())
    print("Command:", " ".join(command))
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
    print(completed.stdout)
    if completed.stderr:
        print("STDERR:\n" + completed.stderr)
    print("GPU after:\n" + gpu_snapshot())
    if completed.returncode:
        raise RuntimeError(f"OCR exited with {completed.returncode}")
    return output

def show_all_pages(result_path):
    payload = json.loads(Path(result_path).read_text(encoding="utf-8"))
    print(json.dumps({"configuration": payload["configuration"], "summary": payload["summary"]}, ensure_ascii=False, indent=2))
    records = sorted(payload["records"], key=lambda item: int(Path(item["images"][0]).stem.split("-")[-1]))
    for record in records:
        page = int(Path(record["images"][0]).stem.split("-")[-1])
        print(f"\n--- OCR output: page {page:02d} ({record['elapsed_seconds']}s) ---")
        print(record.get("text", ""))

def show_gray_audit(longest_edge):
    payload = json.loads(MANIFEST.read_text(encoding="utf-8"))
    records = payload["sizes"][str(longest_edge)]["records"]
    print(json.dumps(records, ensure_ascii=False, indent=2))


In [ ]:
sys.path.insert(0, str(ROOT / "benchmark"))
from ocr_benchmark import PROMPTS

print("Prompt profile:", PROMPT_PROFILE)
print(PROMPTS[PROMPT_PROFILE])
print("\nRun controls:")
print(json.dumps({
    "model": MODEL,
    "endpoint": ENDPOINT,
    "MTP_tokens": MTP_TOKENS,
    "server_max_num_seqs": SERVER_MAX_NUM_SEQS,
    "concurrency": CONCURRENCY,
    "disable_thinking": DISABLE_THINKING,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
}, ensure_ascii=False, indent=2))
wait_for_model()


## 1024px raw

In [1]:
raw_1024 = run_ocr("raw 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024", "results/qwen38_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json")
show_all_pages(raw_1024)


=== raw 1024px ===
Recovered from persisted OCR result: results/qwen38_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json
{
  "configuration": {
    "endpoint": "http://127.0.0.1:8000",
    "model": "qwen3.8-27b",
    "image_directory": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024",
    "image_pattern": "page-*.png",
    "prompt_profile": "document",
    "prompt_plan": null,
    "temperature": 0.0,
    "top_p": 0.8,
    "top_k": 20,
    "repetition_penalty": 1.05,
    "presence_penalty": 0.0,
    "max_tokens": 8192,
    "concurrency": 7,
    "images_per_request": 1,
    "server_max_num_seqs": 7,
    "disable_thinking": true
  },
  "summary": {
    "images": 21,
    "requests": 21,
    "elapsed_seconds": 363.2808,
    "seconds_per_image": 17.2991,
    "completion_tokens": 29485,
    "end_to_end_completion_tokens_per_second": 81.163
  }
}

--- OCR output: page 01 (19.1855s) ---
สำนักงานคณะกรรมการกฤษฎีกา

พระบาทสมเด็จพระเจ้าอยู่หัว
ทรงพระกรุณาโปรดเกล้าฯ ให้ประกา

## 1024px gray220

In [1]:
show_gray_audit(1024)
gray_1024 = run_ocr("gray220 1024px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024", "results/qwen38_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json")
show_all_pages(gray_1024)


Gray220 audit for every page:
[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-01.png",
    "dimensions": [
      724,
      1024
    ],
    "threshold": 220,
    "pixels_replaced": 701722,
    "pixels_total": 741376,
    "pixels_replaced_percent": 94.6513,
    "raw_sha256": "52d5568aaef582dcec78889ffe4cf949683dd20c55c73ad0efb135690563596e",
    "gray220_sha256": "361f82d8dcccbf24b406e0de1d588675d31b4ba41964aeb8dddd18dc65ee73a3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1024/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1024/page-02.png",
    "dimensions": [
      724,
      1024
    ],

## 1400px raw

In [1]:
raw_1400 = run_ocr("raw 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400", "results/qwen38_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json")
show_all_pages(raw_1400)


=== raw 1400px ===
Recovered from persisted OCR result: results/qwen38_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json
{
  "configuration": {
    "endpoint": "http://127.0.0.1:8000",
    "model": "qwen3.8-27b",
    "image_directory": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400",
    "image_pattern": "page-*.png",
    "prompt_profile": "document",
    "prompt_plan": null,
    "temperature": 0.0,
    "top_p": 0.8,
    "top_k": 20,
    "repetition_penalty": 1.05,
    "presence_penalty": 0.0,
    "max_tokens": 8192,
    "concurrency": 7,
    "images_per_request": 1,
    "server_max_num_seqs": 7,
    "disable_thinking": true
  },
  "summary": {
    "images": 21,
    "requests": 21,
    "elapsed_seconds": 356.4993,
    "seconds_per_image": 16.9762,
    "completion_tokens": 23756,
    "end_to_end_completion_tokens_per_second": 66.637
  }
}

--- OCR output: page 01 (20.7915s) ---
สำนักงานคณะกรรมการกฤษฎีกา

สำนักงาน พระราชบัญญัติ
สำนักงานคณะกรรมการกฤษฎีกา
การประกอ

## 1400px gray220

In [1]:
show_gray_audit(1400)
gray_1400 = run_ocr("gray220 1400px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400", "results/qwen38_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json")
show_all_pages(gray_1400)


Gray220 audit for every page:
[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-01.png",
    "dimensions": [
      990,
      1400
    ],
    "threshold": 220,
    "pixels_replaced": 1323612,
    "pixels_total": 1386000,
    "pixels_replaced_percent": 95.4987,
    "raw_sha256": "027ea1c68561df52ce2beea388ee8c612768006d6feb1f4fa09bcabcff165a18",
    "gray220_sha256": "48c9be1e4a099ded0fdef40c00daade1db540a7efd6315a691070fb821bf077c"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1400/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1400/page-02.png",
    "dimensions": [
      990,
      1400
    

## 1600px raw

In [1]:
raw_1600 = run_ocr("raw 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600", "results/qwen38_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json")
show_all_pages(raw_1600)


=== raw 1600px ===
Recovered from persisted OCR result: results/qwen38_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json
{
  "configuration": {
    "endpoint": "http://127.0.0.1:8000",
    "model": "qwen3.8-27b",
    "image_directory": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600",
    "image_pattern": "page-*.png",
    "prompt_profile": "document",
    "prompt_plan": null,
    "temperature": 0.0,
    "top_p": 0.8,
    "top_k": 20,
    "repetition_penalty": 1.05,
    "presence_penalty": 0.0,
    "max_tokens": 8192,
    "concurrency": 7,
    "images_per_request": 1,
    "server_max_num_seqs": 7,
    "disable_thinking": true
  },
  "summary": {
    "images": 21,
    "requests": 21,
    "elapsed_seconds": 70.4536,
    "seconds_per_image": 3.3549,
    "completion_tokens": 11030,
    "end_to_end_completion_tokens_per_second": 156.557
  }
}

--- OCR output: page 01 (21.605s) ---
สำนักงานคณะกรรมการกฤษฎีกา

พระบรมราชโองการ
ประกาศ
พระราชบัญญัติ
การประกอบธุรกิจข้อมูลเ

## 1600px gray220

In [1]:
show_gray_audit(1600)
gray_1600 = run_ocr("gray220 1600px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600", "results/qwen38_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json")
show_all_pages(gray_1600)


Gray220 audit for every page:
[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-01.png",
    "dimensions": [
      1131,
      1600
    ],
    "threshold": 220,
    "pixels_replaced": 1732917,
    "pixels_total": 1809600,
    "pixels_replaced_percent": 95.7624,
    "raw_sha256": "7cd88e5cf12e59d83931179971c152df49e476e3e08d7ba1eb6be1fa0f5d46b7",
    "gray220_sha256": "eaf2a4fa6c8aa8ea8200b12cd7eaaa81dd4aa418aab469cf58f59cbc1d5c13e2"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1600/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1600/page-02.png",
    "dimensions": [
      1131,
      1600
  

## 1800px raw

In [1]:
raw_1800 = run_ocr("raw 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800", "results/qwen38_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json")
show_all_pages(raw_1800)


=== raw 1800px ===
Recovered from persisted OCR result: results/qwen38_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json
{
  "configuration": {
    "endpoint": "http://127.0.0.1:8000",
    "model": "qwen3.8-27b",
    "image_directory": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800",
    "image_pattern": "page-*.png",
    "prompt_profile": "document",
    "prompt_plan": null,
    "temperature": 0.0,
    "top_p": 0.8,
    "top_k": 20,
    "repetition_penalty": 1.05,
    "presence_penalty": 0.0,
    "max_tokens": 8192,
    "concurrency": 7,
    "images_per_request": 1,
    "server_max_num_seqs": 7,
    "disable_thinking": true
  },
  "summary": {
    "images": 21,
    "requests": 21,
    "elapsed_seconds": 77.0747,
    "seconds_per_image": 3.6702,
    "completion_tokens": 11215,
    "end_to_end_completion_tokens_per_second": 145.508
  }
}

--- OCR output: page 01 (19.3595s) ---
สำนักงานคณะกรรมการกฤษฎีกา

พระราชนิพนธ์
การประกอบธุรกิจข้อมูลเครดิต
พ.ศ. ๒๕๔๘

ภูมิพล

## 1800px gray220

In [1]:
show_gray_audit(1800)
gray_1800 = run_ocr("gray220 1800px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800", "results/qwen38_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json")
show_all_pages(gray_1800)


Gray220 audit for every page:
[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-01.png",
    "dimensions": [
      1272,
      1800
    ],
    "threshold": 220,
    "pixels_replaced": 2197434,
    "pixels_total": 2289600,
    "pixels_replaced_percent": 95.9746,
    "raw_sha256": "ea45e6faa8ac8d1bc413f017d0428dae7458faee05babfde03c571c008faf256",
    "gray220_sha256": "e9aaa8c9c0ebf36bd4f65e716f6e8ebe06fa133de12a32db5f5af0df5df13234"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_1800/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_1800/page-02.png",
    "dimensions": [
      1272,
      1800
  

## 2000px raw

In [1]:
raw_2000 = run_ocr("raw 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000", "results/qwen38_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json")
show_all_pages(raw_2000)


=== raw 2000px ===
Recovered from persisted OCR result: results/qwen38_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json
{
  "configuration": {
    "endpoint": "http://127.0.0.1:8000",
    "model": "qwen3.8-27b",
    "image_directory": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000",
    "image_pattern": "page-*.png",
    "prompt_profile": "document",
    "prompt_plan": null,
    "temperature": 0.0,
    "top_p": 0.8,
    "top_k": 20,
    "repetition_penalty": 1.05,
    "presence_penalty": 0.0,
    "max_tokens": 8192,
    "concurrency": 7,
    "images_per_request": 1,
    "server_max_num_seqs": 7,
    "disable_thinking": true
  },
  "summary": {
    "images": 21,
    "requests": 21,
    "elapsed_seconds": 76.6501,
    "seconds_per_image": 3.65,
    "completion_tokens": 11407,
    "end_to_end_completion_tokens_per_second": 148.819
  }
}

--- OCR output: page 01 (20.6895s) ---
สำนักงานคณะกรรมการกฤษฎีกา

พระราชนิพนธ์
การประกอบธุรกิจข้อมูลเครดิต
พ.ศ. ๒๕๔๕

ภูมิพลอด

## 2000px gray220

In [1]:
show_gray_audit(2000)
gray_2000 = run_ocr("gray220 2000px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000", "results/qwen38_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json")
show_all_pages(gray_2000)


Gray220 audit for every page:
[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-01.png",
    "dimensions": [
      1414,
      2000
    ],
    "threshold": 220,
    "pixels_replaced": 2718321,
    "pixels_total": 2828000,
    "pixels_replaced_percent": 96.1217,
    "raw_sha256": "d4a80b77100cb787878ee295a47b493e6c6afbe6b6619cf11bad88dc30e67a46",
    "gray220_sha256": "ee2902a15cdec3354762e8361c56bb13eb98bf8e0675d74bf0f719754b0f9ae3"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2000/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2000/page-02.png",
    "dimensions": [
      1414,
      2000
  

## 2200px raw

In [1]:
raw_2200 = run_ocr("raw 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200", "results/qwen38_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json")
show_all_pages(raw_2200)


=== raw 2200px ===
Recovered from persisted OCR result: results/qwen38_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json
{
  "configuration": {
    "endpoint": "http://127.0.0.1:8000",
    "model": "qwen3.8-27b",
    "image_directory": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200",
    "image_pattern": "page-*.png",
    "prompt_profile": "document",
    "prompt_plan": null,
    "temperature": 0.0,
    "top_p": 0.8,
    "top_k": 20,
    "repetition_penalty": 1.05,
    "presence_penalty": 0.0,
    "max_tokens": 8192,
    "concurrency": 7,
    "images_per_request": 1,
    "server_max_num_seqs": 7,
    "disable_thinking": true
  },
  "summary": {
    "images": 21,
    "requests": 21,
    "elapsed_seconds": 82.1539,
    "seconds_per_image": 3.9121,
    "completion_tokens": 11168,
    "end_to_end_completion_tokens_per_second": 135.94
  }
}

--- OCR output: page 01 (25.3322s) ---
สำนักงานคณะกรรมการกฤษฎีกา

สำนักงานพระราขบัญญัติ
พระราขบัญญัติ
การประกอบธุรกิจข้อมูลเค

## 2200px gray220

In [1]:
show_gray_audit(2200)
gray_2200 = run_ocr("gray220 2200px", "inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200", "results/qwen38_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json")
show_all_pages(gray_2200)


Gray220 audit for every page:
[
  {
    "page": 1,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-01.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-01.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-01.png",
    "dimensions": [
      1555,
      2200
    ],
    "threshold": 220,
    "pixels_replaced": 3302179,
    "pixels_total": 3421000,
    "pixels_replaced_percent": 96.5267,
    "raw_sha256": "289cc186f8272818eebe04ab7817c751400dc99e89e853689e300a2a35c950b1",
    "gray220_sha256": "aca77e3bbbc84f3448de60b79c196caa439ce1ccdefeb9725d48733840f59327"
  },
  {
    "page": 2,
    "source_image": "./inputs/bot_credit_bureau_2559/pages_2200/page-02.png",
    "raw_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/raw_2200/page-02.png",
    "gray220_image": "./inputs/bot_credit_bureau_2559/resolution_sweep/gray220_2200/page-02.png",
    "dimensions": [
      1555,
      2200
  

## Golden score for every size and preprocessing variant

In [1]:
score_path = ROOT / "results/qwen38_resolution_sweep_raw_gray220_20260829.score.json"
score_path.unlink(missing_ok=True)
score_command = [str(AUDIT_PYTHON), "benchmark/score_golden_ocr.py", str(GOLDEN), str(score_path), "--target", "raw_px1024=results/qwen38_bot_credit_bureau_21p_raw_px1024_resolution_sweep_20260829.json", "--target", "gray220_px1024=results/qwen38_bot_credit_bureau_21p_gray220_px1024_resolution_sweep_20260829.json", "--target", "raw_px1400=results/qwen38_bot_credit_bureau_21p_raw_px1400_resolution_sweep_20260829.json", "--target", "gray220_px1400=results/qwen38_bot_credit_bureau_21p_gray220_px1400_resolution_sweep_20260829.json", "--target", "raw_px1600=results/qwen38_bot_credit_bureau_21p_raw_px1600_resolution_sweep_20260829.json", "--target", "gray220_px1600=results/qwen38_bot_credit_bureau_21p_gray220_px1600_resolution_sweep_20260829.json", "--target", "raw_px1800=results/qwen38_bot_credit_bureau_21p_raw_px1800_resolution_sweep_20260829.json", "--target", "gray220_px1800=results/qwen38_bot_credit_bureau_21p_gray220_px1800_resolution_sweep_20260829.json", "--target", "raw_px2000=results/qwen38_bot_credit_bureau_21p_raw_px2000_resolution_sweep_20260829.json", "--target", "gray220_px2000=results/qwen38_bot_credit_bureau_21p_gray220_px2000_resolution_sweep_20260829.json", "--target", "raw_px2200=results/qwen38_bot_credit_bureau_21p_raw_px2200_resolution_sweep_20260829.json", "--target", "gray220_px2200=results/qwen38_bot_credit_bureau_21p_gray220_px2200_resolution_sweep_20260829.json"]
print("Command:", " ".join(score_command))
completed = subprocess.run(score_command, cwd=ROOT, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr:
    print("STDERR:\n" + completed.stderr)
score_payload = json.loads(score_path.read_text(encoding="utf-8"))
for label, target in score_payload["targets"].items():
    summary = target["summary"]
    print("\n", label)
    print(json.dumps({
        "word_accuracy_percent": summary["words"]["accuracy_percent"],
        "word_deltas": {key: summary["words"][key] for key in ("substituted", "missing", "extra")},
        "number_accuracy_percent": summary["number_tokens"]["accuracy_percent"],
    }, ensure_ascii=False, indent=2))


Score written to: results/qwen38_resolution_sweep_raw_gray220_20260829.score.json

raw_px1024
{
  "word_accuracy_percent": 67.327,
  "word_deltas": {
    "substituted": 1656,
    "missing": 587,
    "extra": 2784
  },
  "number_accuracy_percent": 58.791
}

gray220_px1024
{
  "word_accuracy_percent": 67.851,
  "word_deltas": {
    "substituted": 1493,
    "missing": 714,
    "extra": 1413
  },
  "number_accuracy_percent": 56.868
}

raw_px1400
{
  "word_accuracy_percent": 80.583,
  "word_deltas": {
    "substituted": 1194,
    "missing": 139,
    "extra": 3858
  },
  "number_accuracy_percent": 69.505
}

gray220_px1400
{
  "word_accuracy_percent": 85.972,
  "word_deltas": {
    "substituted": 852,
    "missing": 111,
    "extra": 1005
  },
  "number_accuracy_percent": 74.725
}

raw_px1600
{
  "word_accuracy_percent": 90.546,
  "word_deltas": {
    "substituted": 521,
    "missing": 128,
    "extra": 241
  },
  "number_accuracy_percent": 72.527
}

gray220_px1600
{
  "word_accuracy_percent"